Step 07 baseline results
        ↓
Define model-specific tuning spaces
        ↓
Ridge tuning
ElasticNet tuning
SVR tuning
Random Forest complexity control
XGBoost regularization
        ↓
Inner cross-validation
        ↓
Best hyperparameters
        ↓
Before vs After comparison
        ↓
Repeated CV stability check
        ↓
Nested Cross-Validation
        ↓
Final model comparison
        ↓
Select BEST GENERALIZING MODEL
        ↓
Fit exact final pipeline
        ↓
SAVE EXACT FITTED MODEL
        ↓
Save metadata + feature contract

In [1]:
from pathlib import Path

import json
import hashlib
import warnings

import numpy as np
import pandas as pd
import joblib

from scipy.stats import (
    loguniform,
    randint,
    uniform
)

from sklearn.base import clone

from sklearn.model_selection import (
    RepeatedKFold,
    KFold,
    GridSearchCV,
    RandomizedSearchCV,
    cross_validate,
    cross_val_score
)

from sklearn.pipeline import Pipeline

from sklearn.preprocessing import (
    StandardScaler
)

from sklearn.linear_model import (
    LinearRegression,
    Ridge,
    ElasticNet
)

from sklearn.svm import SVR

from sklearn.ensemble import (
    RandomForestRegressor
)

from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error
)

from xgboost import XGBRegressor

warnings.filterwarnings("ignore")

In [2]:
PROJECT_DIR = Path.cwd()

if PROJECT_DIR.name == "phase3_psycap":
    PROJECT_DIR = PROJECT_DIR.parent.parent

DATA_PATH = (
    PROJECT_DIR
    / "data"
    / "processed"
    / "phase3_psycap_prepared.csv"
)

REPORT_DIR = (
    PROJECT_DIR
    / "reports"
    / "phase3"
    / "hyperparameter_tuning"
)

MODEL_DIR = (
    PROJECT_DIR
    / "models"
    / "phase3"
)

REPORT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("PROJECT_DIR:", PROJECT_DIR)
print("DATA_PATH:", DATA_PATH)
print("REPORT_DIR:", REPORT_DIR)
print("MODEL_DIR:", MODEL_DIR)

PROJECT_DIR: d:\internship-employability-project
DATA_PATH: d:\internship-employability-project\data\processed\phase3_psycap_prepared.csv
REPORT_DIR: d:\internship-employability-project\reports\phase3\hyperparameter_tuning
MODEL_DIR: d:\internship-employability-project\models\phase3


In [3]:
df = pd.read_csv(
    DATA_PATH
)

print(
    "Dataset shape:",
    df.shape
)

df.head()

Dataset shape: (1000, 41)


,Response_ID,Gender,Age,Degree Program,Internship Status,PsyCap1,PsyCap2,PsyCap3,PsyCap4,PsyCap5,...,Optimism_avg,Resilience_avg,SelfEfficacy_avg,PsyCap_avg,HumanCapital_avg,SocialCapital_avg,CareerIdentity_avg,EnvironmentalMonitoring_avg,Balance_avg,EVS_avg
0,1,Female,18–20,Information Technology,Yes,5,4,4,4,5,...,4.333333,4.000000,3.666667,4.083333,5.000000,4.0,4.75,5.0,5.0,4.750000
1,2,Female,21–23,Business Management,Yes,3,5,5,4,4,...,4.000000,3.666667,3.666667,3.916667,3.666667,3.5,3.75,4.0,2.5,3.500000
2,3,Male,18–20,Information Technology,Yes,4,5,4,5,4,...,4.333333,3.333333,3.333333,3.833333,3.666667,4.0,3.75,4.0,3.5,3.750000
3,4,Male,18–20,Engineering,Yes,4,4,4,4,4,...,4.000000,4.000000,4.000000,4.000000,4.000000,4.0,4.00,4.0,4.0,4.000000
4,5,Male,21–23,Business Management,Yes,3,3,5,3,5,...,4.000000,3.666667,3.333333,3.666667,4.000000,4.5,4.50,5.0,4.0,4.333333


In [4]:
FEATURES = [
    "Hope_avg",
    "Optimism_avg",
    "Resilience_avg",
    "SelfEfficacy_avg"
]

TARGET = "EVS_avg"

X = df[
    FEATURES
].copy()

y = df[
    TARGET
].copy()

print(
    "Feature shape:",
    X.shape
)

print(
    "Target shape:",
    y.shape
)

Feature shape: (1000, 4)
Target shape: (1000,)


In [5]:
if X.isnull().any().any():
    raise ValueError(
        "Missing values detected in predictors."
    )

if y.isnull().any():
    raise ValueError(
        "Missing values detected in target."
    )

print(
    "Step 08 input validation passed."
)

Step 08 input validation passed.


Step 07 baseline evidence

In [6]:
baseline_results = pd.DataFrame({
    "Model": [
        "Linear_Regression",
        "Ridge",
        "ElasticNet",
        "SVR_RBF",
        "Random_Forest",
        "XGBoost"
    ],

    "Before_CV_R2": [
        0.572709,
        0.572777,
        0.573079,
        0.467615,
        0.476973,
        0.441957
    ],

    "Before_CV_Std": [
        0.096069,
        0.095974,
        0.093070,
        0.088556,
        0.125109,
        0.147217
    ],

    "Before_Gap": [
        0.027480,
        0.027413,
        0.026801,
        0.267291,
        0.357704,
        0.347710
    ]
})

baseline_results

,Model,Before_CV_R2,Before_CV_Std,Before_Gap
0,Linear_Regression,0.572709,0.096069,0.027480
1,Ridge,0.572777,0.095974,0.027413
2,ElasticNet,0.573079,0.093070,0.026801
3,SVR_RBF,0.467615,0.088556,0.267291
4,Random_Forest,0.476973,0.125109,0.357704
5,XGBoost,0.441957,0.147217,0.347710


## Hyperparameter Optimization Strategy

Different algorithms have different causes of overfitting. Therefore, model-specific tuning strategies are used.

### Ridge Regression
Regularization strength (`alpha`) optimize කරයි.

### Elastic Net
Overall regularization strength (`alpha`) සහ L1/L2 balance (`l1_ratio`) optimize කරයි.

### SVR-RBF
Model flexibility control කරන `C`, `gamma` සහ error-tolerance parameter `epsilon` optimize කරයි.

### Random Forest
Tree complexity control කිරීම සඳහා `max_depth`, `min_samples_leaf`, `min_samples_split` සහ `max_features` tune කරයි.

### XGBoost
Tree depth, learning rate, subsampling සහ L1/L2 regularization parameters tune කර overfitting reduce කරයි.

Scaling අවශ්‍ය models සඳහා StandardScaler pipeline එක තුළම භාවිත කරන බැවින් cross-validation leakage වලක්වයි.

Inner CV

In [7]:
INNER_CV = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

Ridge tuning

In [8]:
ridge_pipeline = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "model",
        Ridge()
    )
])

ridge_grid = {
    "model__alpha": np.logspace(
        -4,
        4,
        80
    )
}

ridge_search = GridSearchCV(
    estimator=ridge_pipeline,
    param_grid=ridge_grid,
    scoring="r2",
    cv=INNER_CV,
    n_jobs=-1,
    refit=True
)

ridge_search.fit(
    X,
    y
)

print(
    "Best Ridge parameters:",
    ridge_search.best_params_
)

print(
    "Best Ridge inner-CV R²:",
    ridge_search.best_score_
)

Best Ridge parameters: {'model__alpha': np.float64(119.11031332830055)}
Best Ridge inner-CV R²: 0.5598888666668748


ElasticNet tuning

In [9]:
elastic_pipeline = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "model",
        ElasticNet(
            max_iter=50000,
            random_state=42
        )
    )
])

elastic_grid = {
    "model__alpha": np.logspace(
        -4,
        0,
        30
    ),

    "model__l1_ratio": [
        0.0,
        0.1,
        0.25,
        0.5,
        0.75,
        0.9,
        1.0
    ]
}

elastic_search = GridSearchCV(
    estimator=elastic_pipeline,
    param_grid=elastic_grid,
    scoring="r2",
    cv=INNER_CV,
    n_jobs=-1,
    refit=True
)

elastic_search.fit(
    X,
    y
)

print(
    "Best ElasticNet parameters:",
    elastic_search.best_params_
)

print(
    "Best ElasticNet inner-CV R²:",
    elastic_search.best_score_
)

Best ElasticNet parameters: {'model__alpha': np.float64(0.14873521072935117), 'model__l1_ratio': 0.0}
Best ElasticNet inner-CV R²: 0.5598895881422186


SVR tuning

In [10]:
svr_pipeline = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "model",
        SVR(
            kernel="rbf"
        )
    )
])

svr_param_distributions = {

    "model__C":
        loguniform(
            0.01,
            100
        ),

    "model__gamma":
        loguniform(
            0.0001,
            1.0
        ),

    "model__epsilon":
        uniform(
            0.01,
            0.30
        )
}

svr_search = RandomizedSearchCV(
    estimator=svr_pipeline,
    param_distributions=svr_param_distributions,
    n_iter=60,
    scoring="r2",
    cv=INNER_CV,
    random_state=42,
    n_jobs=-1,
    refit=True
)

svr_search.fit(
    X,
    y
)

print(
    "Best SVR parameters:",
    svr_search.best_params_
)

print(
    "Best SVR inner-CV R²:",
    svr_search.best_score_
)

Best SVR parameters: {'model__C': np.float64(2.481040974867813), 'model__epsilon': np.float64(0.05680559213273095), 'model__gamma': np.float64(0.0004207053950287938)}
Best SVR inner-CV R²: 0.5634688082980619


Random Forest overfitting resolution

## Random Forest Complexity Control

Step 07 එකේ Random Forest high training performance නමුත් substantially lower validation performance පෙන්වූ බැවින් overfitting evidence ලැබුණි.

එබැවින් Step 08 එකේ deep individual trees build කිරීම වෙනුවට:

- maximum tree depth control කිරීම
- minimum leaf size increase කිරීම
- minimum split size tune කිරීම
- feature subsampling
- ensemble size tune කිරීම

මගින් model complexity reduce කිරීමට උත්සාහ කරයි.

In [11]:
rf_pipeline = Pipeline([
    (
        "model",
        RandomForestRegressor(
            random_state=42,
            n_jobs=-1
        )
    )
])

rf_param_distributions = {

    "model__n_estimators": [
        200,
        300,
        500,
        700,
        1000
    ],

    "model__max_depth": [
        2,
        3,
        4,
        5,
        6,
        8,
        10,
        None
    ],

    "model__min_samples_split": [
        2,
        4,
        6,
        10,
        15,
        20
    ],

    "model__min_samples_leaf": [
        1,
        2,
        3,
        4,
        6,
        8,
        12,
        16
    ],

    "model__max_features": [
        0.5,
        0.75,
        1.0,
        "sqrt"
    ],

    "model__bootstrap": [
        True
    ]
}

rf_search = RandomizedSearchCV(
    estimator=rf_pipeline,
    param_distributions=rf_param_distributions,
    n_iter=60,
    scoring="r2",
    cv=INNER_CV,
    random_state=42,
    n_jobs=-1,
    refit=True
)

rf_search.fit(
    X,
    y
)

print(
    "Best Random Forest parameters:",
    rf_search.best_params_
)

print(
    "Best Random Forest inner-CV R²:",
    rf_search.best_score_
)

Best Random Forest parameters: {'model__n_estimators': 300, 'model__min_samples_split': 15, 'model__min_samples_leaf': 6, 'model__max_features': 'sqrt', 'model__max_depth': None, 'model__bootstrap': True}
Best Random Forest inner-CV R²: 0.5423297759401301


XGBoost overfitting resolution

## XGBoost Regularization and Complexity Control

Step 07 XGBoost model එකේ large train-validation gap එක overfitting evidence ලෙස හඳුනා ගන්නා ලදී.

එබැවින් Step 08 එකේ:

- shallow trees
- smaller learning rates
- row subsampling
- feature subsampling
- minimum child-weight control
- L1 regularization
- L2 regularization

මගින් complexity reduce කර validation performance improve කිරීමට උත්සාහ කරයි.

In [12]:
xgb_pipeline = Pipeline([
    (
        "model",
        XGBRegressor(
            objective="reg:squarederror",
            random_state=42,
            n_jobs=-1
        )
    )
])

xgb_param_distributions = {

    "model__n_estimators": [
        100,
        200,
        300,
        500,
        700,
        1000
    ],

    "model__learning_rate":
        loguniform(
            0.005,
            0.20
        ),

    "model__max_depth": [
        2,
        3,
        4,
        5
    ],

    "model__min_child_weight": [
        1,
        2,
        3,
        5,
        8,
        12
    ],

    "model__subsample":
        uniform(
            0.55,
            0.45
        ),

    "model__colsample_bytree":
        uniform(
            0.55,
            0.45
        ),

    "model__reg_alpha":
        loguniform(
            1e-4,
            5
        ),

    "model__reg_lambda":
        loguniform(
            0.1,
            20
        ),

    "model__gamma": [
        0,
        0.01,
        0.05,
        0.1,
        0.25,
        0.5
    ]
}

xgb_search = RandomizedSearchCV(
    estimator=xgb_pipeline,
    param_distributions=xgb_param_distributions,
    n_iter=70,
    scoring="r2",
    cv=INNER_CV,
    random_state=42,
    n_jobs=-1,
    refit=True
)

xgb_search.fit(
    X,
    y
)

print(
    "Best XGBoost parameters:",
    xgb_search.best_params_
)

print(
    "Best XGBoost inner-CV R²:",
    xgb_search.best_score_
)

Best XGBoost parameters: {'model__colsample_bytree': np.float64(0.8066997884824298), 'model__gamma': 0.1, 'model__learning_rate': np.float64(0.005907814282530534), 'model__max_depth': 5, 'model__min_child_weight': 8, 'model__n_estimators': 700, 'model__reg_alpha': np.float64(0.001883913511943895), 'model__reg_lambda': np.float64(8.071418522169695), 'model__subsample': np.float64(0.7105389970121152)}
Best XGBoost inner-CV R²: 0.5182885057970725


Collect tuned estimators

In [13]:
tuned_models = {

    "Linear_Regression":
        Pipeline([
            (
                "scaler",
                StandardScaler()
            ),
            (
                "model",
                LinearRegression()
            )
        ]),

    "Ridge_Tuned":
        ridge_search.best_estimator_,

    "ElasticNet_Tuned":
        elastic_search.best_estimator_,

    "SVR_RBF_Tuned":
        svr_search.best_estimator_,

    "Random_Forest_Tuned":
        rf_search.best_estimator_,

    "XGBoost_Tuned":
        xgb_search.best_estimator_
}

Repeated CV after tuning

In [14]:
REPEATED_CV = RepeatedKFold(
    n_splits=5,
    n_repeats=10,
    random_state=42
)

In [15]:
tuned_cv_results = []

tuned_model_r2_scores = {}

for model_name, estimator in tuned_models.items():

    print(
        "Evaluating:",
        model_name
    )

    result = cross_validate(
        estimator,
        X,
        y,
        cv=REPEATED_CV,
        scoring={
            "R2":
                "r2",

            "MAE":
                "neg_mean_absolute_error",

            "RMSE":
                "neg_root_mean_squared_error"
        },
        return_train_score=True,
        n_jobs=-1
    )

    test_r2 = result[
        "test_R2"
    ]

    train_r2 = result[
        "train_R2"
    ]

    tuned_model_r2_scores[
        model_name
    ] = test_r2.copy()

    tuned_cv_results.append({

        "Model":
            model_name,

        "CV_Mean_R2":
            test_r2.mean(),

        "CV_Median_R2":
            np.median(
                test_r2
            ),

        "CV_Std_R2":
            test_r2.std(),

        "CV_Min_R2":
            test_r2.min(),

        "CV_Max_R2":
            test_r2.max(),

        "CV_R2_2.5_Percentile":
            np.percentile(
                test_r2,
                2.5
            ),

        "CV_R2_97.5_Percentile":
            np.percentile(
                test_r2,
                97.5
            ),

        "CV_Mean_Train_R2":
            train_r2.mean(),

        "CV_Train_Validation_Gap":
            (
                train_r2.mean()
                -
                test_r2.mean()
            ),

        "CV_Mean_MAE":
            -result[
                "test_MAE"
            ].mean(),

        "CV_Mean_RMSE":
            -result[
                "test_RMSE"
            ].mean()
    })


tuned_cv_results_df = pd.DataFrame(
    tuned_cv_results
)

tuned_cv_results_df.sort_values(
    "CV_Mean_R2",
    ascending=False
)

Evaluating: Linear_Regression
Evaluating: Ridge_Tuned
Evaluating: ElasticNet_Tuned
Evaluating: SVR_RBF_Tuned
Evaluating: Random_Forest_Tuned
Evaluating: XGBoost_Tuned


,Model,CV_Mean_R2,CV_Median_R2,CV_Std_R2,CV_Min_R2,CV_Max_R2,CV_R2_2.5_Percentile,CV_R2_97.5_Percentile,CV_Mean_Train_R2,CV_Train_Validation_Gap,CV_Mean_MAE,CV_Mean_RMSE
3,SVR_RBF_Tuned,0.577473,0.579881,0.088256,0.329203,0.749977,0.406525,0.728435,0.593188,0.015715,0.136862,0.228607
2,ElasticNet_Tuned,0.574259,0.577918,0.088289,0.326600,0.741412,0.399285,0.716122,0.596255,0.021995,0.136181,0.229558
1,Ridge_Tuned,0.574256,0.577917,0.088282,0.326601,0.741396,0.399309,0.716107,0.596248,0.021992,0.136182,0.229559
0,Linear_Regression,0.572709,0.575226,0.096069,0.333303,0.750842,0.364993,0.726162,0.600190,0.027480,0.135626,0.229577
4,Random_Forest_Tuned,0.557641,0.568106,0.084422,0.289837,0.707714,0.347176,0.680695,0.631512,0.073872,0.132144,0.234538
5,XGBoost_Tuned,0.541806,0.558782,0.086929,0.268478,0.687374,0.301144,0.656547,0.625870,0.084064,0.135142,0.238770


Overfitting status after tuning

In [16]:
def classify_overfitting(gap):

    if gap <= 0.05:
        return "Low"

    elif gap <= 0.15:
        return "Moderate"

    else:
        return "High"


tuned_cv_results_df[
    "Overfitting_Risk"
] = (
    tuned_cv_results_df[
        "CV_Train_Validation_Gap"
    ]
    .apply(
        classify_overfitting
    )
)

tuned_cv_results_df[
    [
        "Model",
        "CV_Mean_R2",
        "CV_Std_R2",
        "CV_Mean_Train_R2",
        "CV_Train_Validation_Gap",
        "Overfitting_Risk"
    ]
].sort_values(
    "CV_Mean_R2",
    ascending=False
)

,Model,CV_Mean_R2,CV_Std_R2,CV_Mean_Train_R2,CV_Train_Validation_Gap,Overfitting_Risk
3,SVR_RBF_Tuned,0.577473,0.088256,0.593188,0.015715,Low
2,ElasticNet_Tuned,0.574259,0.088289,0.596255,0.021995,Low
1,Ridge_Tuned,0.574256,0.088282,0.596248,0.021992,Low
0,Linear_Regression,0.572709,0.096069,0.600190,0.027480,Low
4,Random_Forest_Tuned,0.557641,0.084422,0.631512,0.073872,Moderate
5,XGBoost_Tuned,0.541806,0.086929,0.625870,0.084064,Moderate


Before vs After problem-resolution table

In [17]:
model_name_mapping = {
    "Ridge_Tuned":
        "Ridge",

    "ElasticNet_Tuned":
        "ElasticNet",

    "SVR_RBF_Tuned":
        "SVR_RBF",

    "Random_Forest_Tuned":
        "Random_Forest",

    "XGBoost_Tuned":
        "XGBoost"
}


before_after_rows = []


for tuned_name, original_name in (
    model_name_mapping.items()
):

    before_row = baseline_results[
        baseline_results[
            "Model"
        ]
        ==
        original_name
    ].iloc[0]

    after_row = tuned_cv_results_df[
        tuned_cv_results_df[
            "Model"
        ]
        ==
        tuned_name
    ].iloc[0]

    before_after_rows.append({

        "Model":
            original_name,

        "Before_CV_R2":
            before_row[
                "Before_CV_R2"
            ],

        "After_CV_R2":
            after_row[
                "CV_Mean_R2"
            ],

        "R2_Improvement":
            (
                after_row[
                    "CV_Mean_R2"
                ]
                -
                before_row[
                    "Before_CV_R2"
                ]
            ),

        "Before_CV_Std":
            before_row[
                "Before_CV_Std"
            ],

        "After_CV_Std":
            after_row[
                "CV_Std_R2"
            ],

        "Before_Gap":
            before_row[
                "Before_Gap"
            ],

        "After_Gap":
            after_row[
                "CV_Train_Validation_Gap"
            ],

        "Gap_Reduction":
            (
                before_row[
                    "Before_Gap"
                ]
                -
                after_row[
                    "CV_Train_Validation_Gap"
                ]
            ),

        "After_Overfitting_Risk":
            after_row[
                "Overfitting_Risk"
            ]
    })


before_after_df = pd.DataFrame(
    before_after_rows
)

before_after_df.sort_values(
    "R2_Improvement",
    ascending=False
)

,Model,Before_CV_R2,After_CV_R2,R2_Improvement,Before_CV_Std,After_CV_Std,Before_Gap,After_Gap,Gap_Reduction,After_Overfitting_Risk
2,SVR_RBF,0.467615,0.577473,0.109858,0.088556,0.088256,0.267291,0.015715,0.251576,Low
4,XGBoost,0.441957,0.541806,0.099849,0.147217,0.086929,0.347710,0.084064,0.263646,Moderate
3,Random_Forest,0.476973,0.557641,0.080668,0.125109,0.084422,0.357704,0.073872,0.283832,Moderate
0,Ridge,0.572777,0.574256,0.001479,0.095974,0.088282,0.027413,0.021992,0.005421,Low
1,ElasticNet,0.573079,0.574259,0.001180,0.093070,0.088289,0.026801,0.021995,0.004806,Low


## Nested Cross-Validation

Hyperparameter tuning සහ model evaluation එකම validation results මත සිදු කළහොත් optimistic performance estimates ඇති විය හැක.

එබැවින් final model-selection stage එකේ nested cross-validation භාවිත කරයි.

Nested CV structure:

- Outer 5-fold CV — unbiased model evaluation
- Inner 5-fold CV — hyperparameter optimization

Outer validation fold එක inner hyperparameter selection process එකෙන් completely separate බැවින් tuned model generalization එක වඩා defensible ලෙස estimate කළ හැක.

Outer CV

In [18]:
OUTER_CV = KFold(
    n_splits=5,
    shuffle=True,
    random_state=2026
)

Refined Ridge nested search

In [19]:
ridge_nested = GridSearchCV(
    estimator=ridge_pipeline,

    param_grid={
        "model__alpha":
            np.logspace(
                -3,
                3,
                30
            )
    },

    scoring="r2",
    cv=INNER_CV,
    n_jobs=-1
)

Refined ElasticNet nested search

In [20]:
elastic_nested = RandomizedSearchCV(
    estimator=elastic_pipeline,

    param_distributions={
        "model__alpha":
            loguniform(
                1e-4,
                1.0
            ),

        "model__l1_ratio":
            uniform(
                0.0,
                1.0
            )
    },

    n_iter=30,
    scoring="r2",
    cv=INNER_CV,
    random_state=42,
    n_jobs=-1
)

Refined SVR nested search

In [21]:
svr_nested = RandomizedSearchCV(
    estimator=svr_pipeline,

    param_distributions={
        "model__C":
            loguniform(
                0.01,
                50
            ),

        "model__gamma":
            loguniform(
                1e-4,
                0.5
            ),

        "model__epsilon":
            uniform(
                0.01,
                0.25
            )
    },

    n_iter=35,
    scoring="r2",
    cv=INNER_CV,
    random_state=42,
    n_jobs=-1
)

Refined RF nested search

In [22]:
rf_nested = RandomizedSearchCV(
    estimator=rf_pipeline,

    param_distributions={
        "model__n_estimators": [
            300,
            500,
            700
        ],

        "model__max_depth": [
            2,
            3,
            4,
            5,
            6,
            8
        ],

        "model__min_samples_leaf": [
            2,
            4,
            6,
            8,
            12,
            16
        ],

        "model__min_samples_split": [
            4,
            6,
            10,
            15,
            20
        ],

        "model__max_features": [
            0.5,
            0.75,
            1.0,
            "sqrt"
        ]
    },

    n_iter=35,
    scoring="r2",
    cv=INNER_CV,
    random_state=42,
    n_jobs=-1
)

Refined XGBoost nested search

In [23]:
xgb_nested = RandomizedSearchCV(
    estimator=xgb_pipeline,

    param_distributions={
        "model__n_estimators": [
            200,
            300,
            500,
            700
        ],

        "model__learning_rate":
            loguniform(
                0.01,
                0.12
            ),

        "model__max_depth": [
            2,
            3,
            4
        ],

        "model__min_child_weight": [
            2,
            3,
            5,
            8,
            12
        ],

        "model__subsample":
            uniform(
                0.6,
                0.4
            ),

        "model__colsample_bytree":
            uniform(
                0.6,
                0.4
            ),

        "model__reg_alpha":
            loguniform(
                1e-4,
                3
            ),

        "model__reg_lambda":
            loguniform(
                0.2,
                15
            )
    },

    n_iter=40,
    scoring="r2",
    cv=INNER_CV,
    random_state=42,
    n_jobs=-1
)

In [24]:
linear_benchmark = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "model",
        LinearRegression()
    )
])

Nested model dictionary

In [25]:
nested_models = {

    "Linear_Regression":
        linear_benchmark,

    "Ridge":
        ridge_nested,

    "ElasticNet":
        elastic_nested,

    "SVR_RBF":
        svr_nested,

    "Random_Forest":
        rf_nested,

    "XGBoost":
        xgb_nested
}

Run Nested CV

In [26]:
nested_results = []

nested_score_store = {}


for model_name, estimator in (
    nested_models.items()
):

    print(
        f"\nNested CV: {model_name}"
    )

    scores = cross_val_score(
        estimator,
        X,
        y,
        scoring="r2",
        cv=OUTER_CV,
        n_jobs=-1
    )

    nested_score_store[
        model_name
    ] = scores.copy()

    nested_results.append({

        "Model":
            model_name,

        "Nested_Mean_R2":
            scores.mean(),

        "Nested_Median_R2":
            np.median(
                scores
            ),

        "Nested_Std_R2":
            scores.std(),

        "Nested_Min_R2":
            scores.min(),

        "Nested_Max_R2":
            scores.max()
    })


nested_results_df = pd.DataFrame(
    nested_results
)

nested_results_df.sort_values(
    "Nested_Mean_R2",
    ascending=False
)


Nested CV: Linear_Regression

Nested CV: Ridge

Nested CV: ElasticNet

Nested CV: SVR_RBF

Nested CV: Random_Forest

Nested CV: XGBoost


,Model,Nested_Mean_R2,Nested_Median_R2,Nested_Std_R2,Nested_Min_R2,Nested_Max_R2
2,ElasticNet,0.573876,0.590726,0.077173,0.468608,0.696599
1,Ridge,0.568767,0.587792,0.087150,0.436727,0.700534
0,Linear_Regression,0.567484,0.592599,0.088107,0.445514,0.701619
3,SVR_RBF,0.566311,0.586362,0.088557,0.425766,0.695761
4,Random_Forest,0.533616,0.554544,0.070255,0.411304,0.599409
5,XGBoost,0.512386,0.550733,0.091586,0.390288,0.628061


Combine repeated CV + nested CV

In [27]:
name_conversion = {
    "Ridge_Tuned":
        "Ridge",

    "ElasticNet_Tuned":
        "ElasticNet",

    "SVR_RBF_Tuned":
        "SVR_RBF",

    "Random_Forest_Tuned":
        "Random_Forest",

    "XGBoost_Tuned":
        "XGBoost",

    "Linear_Regression":
        "Linear_Regression"
}


tuned_for_merge = (
    tuned_cv_results_df
    .copy()
)

tuned_for_merge[
    "Model_Base"
] = (
    tuned_for_merge[
        "Model"
    ]
    .map(
        name_conversion
    )
)


final_selection_table = (
    nested_results_df
    .merge(
        tuned_for_merge,
        left_on="Model",
        right_on="Model_Base",
        how="left",
        suffixes=(
            "_Nested",
            "_Repeated"
        )
    )
)

final_selection_table

,Model_Nested,Nested_Mean_R2,Nested_Median_R2,Nested_Std_R2,Nested_Min_R2,Nested_Max_R2,Model_Repeated,CV_Mean_R2,CV_Median_R2,CV_Std_R2,CV_Min_R2,CV_Max_R2,CV_R2_2.5_Percentile,CV_R2_97.5_Percentile,CV_Mean_Train_R2,CV_Train_Validation_Gap,CV_Mean_MAE,CV_Mean_RMSE,Overfitting_Risk,Model_Base
0,Linear_Regression,0.567484,0.592599,0.088107,0.445514,0.701619,Linear_Regression,0.572709,0.575226,0.096069,0.333303,0.750842,0.364993,0.726162,0.600190,0.027480,0.135626,0.229577,Low,Linear_Regression
1,Ridge,0.568767,0.587792,0.087150,0.436727,0.700534,Ridge_Tuned,0.574256,0.577917,0.088282,0.326601,0.741396,0.399309,0.716107,0.596248,0.021992,0.136182,0.229559,Low,Ridge
2,ElasticNet,0.573876,0.590726,0.077173,0.468608,0.696599,ElasticNet_Tuned,0.574259,0.577918,0.088289,0.326600,0.741412,0.399285,0.716122,0.596255,0.021995,0.136181,0.229558,Low,ElasticNet
3,SVR_RBF,0.566311,0.586362,0.088557,0.425766,0.695761,SVR_RBF_Tuned,0.577473,0.579881,0.088256,0.329203,0.749977,0.406525,0.728435,0.593188,0.015715,0.136862,0.228607,Low,SVR_RBF
4,Random_Forest,0.533616,0.554544,0.070255,0.411304,0.599409,Random_Forest_Tuned,0.557641,0.568106,0.084422,0.289837,0.707714,0.347176,0.680695,0.631512,0.073872,0.132144,0.234538,Moderate,Random_Forest
5,XGBoost,0.512386,0.550733,0.091586,0.390288,0.628061,XGBoost_Tuned,0.541806,0.558782,0.086929,0.268478,0.687374,0.301144,0.656547,0.625870,0.084064,0.135142,0.238770,Moderate,XGBoost


Final ranking

In [28]:
final_selection_table[
    "Nested_R2_Rank"
] = (
    final_selection_table[
        "Nested_Mean_R2"
    ]
    .rank(
        ascending=False,
        method="min"
    )
    .astype(int)
)

final_selection_table[
    "Repeated_R2_Rank"
] = (
    final_selection_table[
        "CV_Mean_R2"
    ]
    .rank(
        ascending=False,
        method="min"
    )
    .astype(int)
)

final_selection_table.sort_values(
    [
        "Nested_R2_Rank",
        "Repeated_R2_Rank"
    ]
)

,Model_Nested,Nested_Mean_R2,Nested_Median_R2,Nested_Std_R2,Nested_Min_R2,Nested_Max_R2,Model_Repeated,CV_Mean_R2,CV_Median_R2,CV_Std_R2,...,CV_R2_2.5_Percentile,CV_R2_97.5_Percentile,CV_Mean_Train_R2,CV_Train_Validation_Gap,CV_Mean_MAE,CV_Mean_RMSE,Overfitting_Risk,Model_Base,Nested_R2_Rank,Repeated_R2_Rank
2,ElasticNet,0.573876,0.590726,0.077173,0.468608,0.696599,ElasticNet_Tuned,0.574259,0.577918,0.088289,...,0.399285,0.716122,0.596255,0.021995,0.136181,0.229558,Low,ElasticNet,1,2
1,Ridge,0.568767,0.587792,0.087150,0.436727,0.700534,Ridge_Tuned,0.574256,0.577917,0.088282,...,0.399309,0.716107,0.596248,0.021992,0.136182,0.229559,Low,Ridge,2,3
0,Linear_Regression,0.567484,0.592599,0.088107,0.445514,0.701619,Linear_Regression,0.572709,0.575226,0.096069,...,0.364993,0.726162,0.600190,0.027480,0.135626,0.229577,Low,Linear_Regression,3,4
3,SVR_RBF,0.566311,0.586362,0.088557,0.425766,0.695761,SVR_RBF_Tuned,0.577473,0.579881,0.088256,...,0.406525,0.728435,0.593188,0.015715,0.136862,0.228607,Low,SVR_RBF,4,1
4,Random_Forest,0.533616,0.554544,0.070255,0.411304,0.599409,Random_Forest_Tuned,0.557641,0.568106,0.084422,...,0.347176,0.680695,0.631512,0.073872,0.132144,0.234538,Moderate,Random_Forest,5,5
5,XGBoost,0.512386,0.550733,0.091586,0.390288,0.628061,XGBoost_Tuned,0.541806,0.558782,0.086929,...,0.301144,0.656547,0.625870,0.084064,0.135142,0.238770,Moderate,XGBoost,6,6


Selection criteria

In [29]:
candidate_selection_view = (
    final_selection_table[
        [
            "Model_Nested",
            "Nested_Mean_R2",
            "Nested_Std_R2",
            "CV_Mean_R2",
            "CV_Std_R2",
            "CV_Mean_MAE",
            "CV_Mean_RMSE",
            "CV_Train_Validation_Gap",
            "Overfitting_Risk",
            "Nested_R2_Rank"
        ]
    ]
    .sort_values(
        "Nested_R2_Rank"
    )
)

candidate_selection_view

,Model_Nested,Nested_Mean_R2,Nested_Std_R2,CV_Mean_R2,CV_Std_R2,CV_Mean_MAE,CV_Mean_RMSE,CV_Train_Validation_Gap,Overfitting_Risk,Nested_R2_Rank
2,ElasticNet,0.573876,0.077173,0.574259,0.088289,0.136181,0.229558,0.021995,Low,1
1,Ridge,0.568767,0.087150,0.574256,0.088282,0.136182,0.229559,0.021992,Low,2
0,Linear_Regression,0.567484,0.088107,0.572709,0.096069,0.135626,0.229577,0.027480,Low,3
3,SVR_RBF,0.566311,0.088557,0.577473,0.088256,0.136862,0.228607,0.015715,Low,4
4,Random_Forest,0.533616,0.070255,0.557641,0.084422,0.132144,0.234538,0.073872,Moderate,5
5,XGBoost,0.512386,0.091586,0.541806,0.086929,0.135142,0.238770,0.084064,Moderate,6


Identify final winner

In [30]:
eligible_models = (
    candidate_selection_view[
        candidate_selection_view[
            "Overfitting_Risk"
        ]
        !=
        "High"
    ]
    .copy()
)

if len(
    eligible_models
) == 0:

    eligible_models = (
        candidate_selection_view
        .copy()
    )


final_winner_row = (
    eligible_models
    .sort_values(
        [
            "Nested_Mean_R2",
            "CV_Mean_RMSE",
            "CV_Std_R2"
        ],
        ascending=[
            False,
            True,
            True
        ]
    )
    .iloc[0]
)


FINAL_MODEL_NAME = (
    final_winner_row[
        "Model_Nested"
    ]
)

print(
    "FINAL SELECTED MODEL:",
    FINAL_MODEL_NAME
)

print(
    "\nSelection evidence:"
)

print(
    final_winner_row
)

FINAL SELECTED MODEL: ElasticNet

Selection evidence:
Model_Nested               ElasticNet
Nested_Mean_R2               0.573876
Nested_Std_R2                0.077173
CV_Mean_R2                   0.574259
CV_Std_R2                    0.088289
CV_Mean_MAE                  0.136181
CV_Mean_RMSE                 0.229558
CV_Train_Validation_Gap      0.021995
Overfitting_Risk                  Low
Nested_R2_Rank                      1
Name: 2, dtype: object


Get exact best estimator

In [31]:
final_estimator_map = {

    "Linear_Regression":
        linear_benchmark,

    "Ridge":
        ridge_search.best_estimator_,

    "ElasticNet":
        elastic_search.best_estimator_,

    "SVR_RBF":
        svr_search.best_estimator_,

    "Random_Forest":
        rf_search.best_estimator_,

    "XGBoost":
        xgb_search.best_estimator_
}

Fit final deployment model ONCE

In [32]:
final_model = clone(
    final_estimator_map[
        FINAL_MODEL_NAME
    ]
)

final_model.fit(
    X,
    y
)

print(
    "Final model fitted successfully."
)

print(
    final_model
)

Final model fitted successfully.
Pipeline(steps=[('scaler', StandardScaler()),
                ('model',
                 ElasticNet(alpha=np.float64(0.14873521072935117), l1_ratio=0.0,
                            max_iter=50000, random_state=42))])


Save exact fitted model

In [33]:
FINAL_MODEL_PATH = (
    MODEL_DIR
    / "phase3_psycap_selected_model.pkl"
)

joblib.dump(
    final_model,
    FINAL_MODEL_PATH
)

print(
    "Saved final model:",
    FINAL_MODEL_PATH
)

Saved final model: d:\internship-employability-project\models\phase3\phase3_psycap_selected_model.pkl


Reload consistency test

In [34]:
reloaded_model = joblib.load(
    FINAL_MODEL_PATH
)

original_predictions = (
    final_model.predict(
        X
    )
)

reloaded_predictions = (
    reloaded_model.predict(
        X
    )
)

max_prediction_difference = np.max(
    np.abs(
        original_predictions
        -
        reloaded_predictions
    )
)

print(
    "Predictions identical:",
    np.allclose(
        original_predictions,
        reloaded_predictions
    )
)

print(
    "Maximum difference:",
    max_prediction_difference
)

Predictions identical: True
Maximum difference: 0.0


In [35]:
def calculate_sha256(
    file_path
):

    sha256 = hashlib.sha256()

    with open(
        file_path,
        "rb"
    ) as file:

        for block in iter(
            lambda:
                file.read(
                    4096
                ),
            b""
        ):

            sha256.update(
                block
            )

    return sha256.hexdigest()


model_sha256 = calculate_sha256(
    FINAL_MODEL_PATH
)

print(
    "Model SHA256:",
    model_sha256
)

Model SHA256: 6d8ed0af743a35f91dccf7f85630056d5a9afb551f2b887b3ff5f4daca5aa75b


Save feature schema

In [36]:
feature_schema = {
    "phase":
        "Phase 03",

    "research_component":
        "Psychological Capital -> Employability",

    "target":
        TARGET,

    "features":
        FEATURES,

    "feature_order":
        FEATURES,

    "number_of_features":
        len(
            FEATURES
        ),

    "input_scale":
        "Derived averages from Likert 1-5 items",

    "expected_feature_range": {
        feature: {
            "min": 1.0,
            "max": 5.0
        }
        for feature in FEATURES
    }
}


FEATURE_SCHEMA_PATH = (
    MODEL_DIR
    / "phase3_psycap_feature_schema.json"
)


with open(
    FEATURE_SCHEMA_PATH,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        feature_schema,
        file,
        indent=4
    )


print(
    "Feature schema saved."
)

Feature schema saved.


Save model metadata

In [37]:
metadata = {

    "phase":
        "Phase 03",

    "model_name":
        FINAL_MODEL_NAME,

    "target":
        TARGET,

    "features":
        FEATURES,

    "selection_method":
        "Nested cross-validation supported by repeated cross-validation",

    "nested_cv_mean_r2":
        float(
            final_winner_row[
                "Nested_Mean_R2"
            ]
        ),

    "nested_cv_std_r2":
        float(
            final_winner_row[
                "Nested_Std_R2"
            ]
        ),

    "repeated_cv_mean_r2":
        float(
            final_winner_row[
                "CV_Mean_R2"
            ]
        ),

    "repeated_cv_std_r2":
        float(
            final_winner_row[
                "CV_Std_R2"
            ]
        ),

    "cv_mean_mae":
        float(
            final_winner_row[
                "CV_Mean_MAE"
            ]
        ),

    "cv_mean_rmse":
        float(
            final_winner_row[
                "CV_Mean_RMSE"
            ]
        ),

    "train_validation_gap":
        float(
            final_winner_row[
                "CV_Train_Validation_Gap"
            ]
        ),

    "model_sha256":
        model_sha256,

    "retraining_policy":
        "Do not retrain in Steps 09-12. Load this exact fitted artifact."
}


METADATA_PATH = (
    MODEL_DIR
    / "phase3_psycap_model_metadata.json"
)


with open(
    METADATA_PATH,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        metadata,
        file,
        indent=4
    )


print(
    "Model metadata saved."
)

Model metadata saved.


Save reference predictions

In [38]:
reference_indices = (
    X.index[
        :20
    ]
)

reference_predictions_df = (
    X.loc[
        reference_indices
    ]
    .copy()
)

reference_predictions_df[
    "Reference_Prediction"
] = (
    final_model.predict(
        X.loc[
            reference_indices
        ]
    )
)

reference_predictions_df.to_csv(
    MODEL_DIR
    / "phase3_psycap_reference_predictions.csv",
    index=False
)

reference_predictions_df

,Hope_avg,Optimism_avg,Resilience_avg,SelfEfficacy_avg,Reference_Prediction
0,4.333333,4.333333,4.000000,3.666667,3.931926
1,4.333333,4.000000,3.666667,3.666667,3.818310
2,4.333333,4.333333,3.333333,3.333333,3.680008
3,4.000000,4.000000,4.000000,4.000000,3.963109
4,3.666667,4.000000,3.666667,3.333333,3.669528
5,3.333333,5.000000,5.000000,4.666667,4.434873
6,5.000000,4.333333,4.333333,4.666667,4.346514
7,3.666667,3.666667,3.666667,3.333333,3.635254
8,4.000000,4.666667,3.666667,4.333333,4.045545
9,4.000000,4.000000,4.000000,4.000000,3.963109


Save best parameters

In [39]:
best_parameter_summary = {

    "Ridge":
        ridge_search.best_params_,

    "ElasticNet":
        elastic_search.best_params_,

    "SVR_RBF":
        svr_search.best_params_,

    "Random_Forest":
        rf_search.best_params_,

    "XGBoost":
        xgb_search.best_params_
}


BEST_PARAMS_PATH = (
    MODEL_DIR
    / "phase3_tuned_hyperparameters.json"
)


with open(
    BEST_PARAMS_PATH,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        best_parameter_summary,
        file,
        indent=4,
        default=str
    )


print(
    "Best hyperparameters saved."
)

Best hyperparameters saved.


In [40]:
baseline_results.to_csv(
    REPORT_DIR
    / "Step07_Baseline_Results.csv",
    index=False
)

tuned_cv_results_df.to_csv(
    REPORT_DIR
    / "Tuned_Model_RepeatedCV_Results.csv",
    index=False
)

before_after_df.to_csv(
    REPORT_DIR
    / "Before_After_Tuning_Comparison.csv",
    index=False
)

nested_results_df.to_csv(
    REPORT_DIR
    / "Nested_CV_Model_Comparison.csv",
    index=False
)

final_selection_table.to_csv(
    REPORT_DIR
    / "Final_Model_Selection_Evidence.csv",
    index=False
)

candidate_selection_view.to_csv(
    REPORT_DIR
    / "Final_Model_Candidate_View.csv",
    index=False
)

print(
    "Step 08 reports saved."
)

Step 08 reports saved.


In [41]:
step08_validation = pd.DataFrame({

    "Check": [

        "Ridge hyperparameters tuned",

        "ElasticNet hyperparameters tuned",

        "SVR hyperparameters tuned",

        "Random Forest complexity tuned",

        "XGBoost regularization tuned",

        "Repeated CV after tuning completed",

        "Before-after tuning comparison completed",

        "Overfitting risk reassessed",

        "Nested cross-validation completed",

        "Final model selected using generalization evidence",

        "Final model fitted",

        "Exact fitted model saved",

        "Saved model successfully reloaded",

        "Reloaded predictions identical",

        "Model hash generated",

        "Feature schema saved",

        "Model metadata saved",

        "Reference predictions saved"
    ],

    "Status": [

        "PASS",
        "PASS",
        "PASS",
        "PASS",
        "PASS",
        "PASS",
        "PASS",
        "PASS",
        "PASS",
        "PASS",
        "PASS",
        "PASS",
        "PASS",

        (
            "PASS"
            if np.allclose(
                original_predictions,
                reloaded_predictions
            )
            else "FAIL"
        ),

        "PASS",
        "PASS",
        "PASS",
        "PASS"
    ]
})

step08_validation

,Check,Status
0,Ridge hyperparameters tuned,PASS
1,ElasticNet hyperparameters tuned,PASS
2,SVR hyperparameters tuned,PASS
3,Random Forest complexity tuned,PASS
4,XGBoost regularization tuned,PASS
5,Repeated CV after tuning completed,PASS
6,Before-after tuning comparison completed,PASS
7,Overfitting risk reassessed,PASS
8,Nested cross-validation completed,PASS
9,Final model selected using generalization evid...,PASS


In [42]:
step08_validation.to_csv(
    REPORT_DIR
    / "Step08_Validation.csv",
    index=False
)

In [43]:
print(
    "PHASE 03 — STEP 08"
)

print(
    "\nBEFORE vs AFTER TUNING:"
)

print(
    before_after_df
    .sort_values(
        "After_CV_R2",
        ascending=False
    )
    .to_string(
        index=False
    )
)

print(
    "\nNESTED CV:"
)

print(
    nested_results_df
    .sort_values(
        "Nested_Mean_R2",
        ascending=False
    )
    .to_string(
        index=False
    )
)

print(
    "\nFINAL SELECTED MODEL:"
)

print(
    FINAL_MODEL_NAME
)

print(
    "\nMODEL PATH:"
)

print(
    FINAL_MODEL_PATH
)

print(
    "\nMODEL SHA256:"
)

print(
    model_sha256
)

print(
    "\nVALIDATION:"
)

print(
    step08_validation
    .to_string(
        index=False
    )
)

PHASE 03 — STEP 08

BEFORE vs AFTER TUNING:
        Model  Before_CV_R2  After_CV_R2  R2_Improvement  Before_CV_Std  After_CV_Std  Before_Gap  After_Gap  Gap_Reduction After_Overfitting_Risk
      SVR_RBF      0.467615     0.577473        0.109858       0.088556      0.088256    0.267291   0.015715       0.251576                    Low
   ElasticNet      0.573079     0.574259        0.001180       0.093070      0.088289    0.026801   0.021995       0.004806                    Low
        Ridge      0.572777     0.574256        0.001479       0.095974      0.088282    0.027413   0.021992       0.005421                    Low
Random_Forest      0.476973     0.557641        0.080668       0.125109      0.084422    0.357704   0.073872       0.283832               Moderate
      XGBoost      0.441957     0.541806        0.099849       0.147217      0.086929    0.347710   0.084064       0.263646               Moderate

NESTED CV:
            Model  Nested_Mean_R2  Nested_Median_R2  Nested_St

## Step 08 අවසාන නිගමනය

Step 07 එකේ හඳුනාගත් model-generalization සහ overfitting issues විසඳීම සඳහා model-specific hyperparameter optimization සහ complexity-control methods භාවිත කරන ලදී.

Ridge සහ Elastic Net models සඳහා regularization strength optimize කරන ලදී. SVR-RBF model සඳහා C, gamma සහ epsilon parameters tune කර nonlinear model flexibility control කරන ලදී.

Random Forest model එකේ tree depth, minimum leaf size, minimum split size සහ feature sampling parameters tune කර overfitting reduce කිරීමට උත්සාහ කරන ලදී. XGBoost model සඳහා tree complexity, learning rate, subsampling සහ L1/L2 regularization optimize කරන ලදී.

Tuning වලට පෙර සහ පසු mean cross-validated R², variability සහ train-validation gap compare කර corrective actions model generalization improve කළේද යන්න evaluate කරන ලදී.

Hyperparameter selection සහ performance evaluation අතර optimistic bias reduce කිරීම සඳහා nested cross-validation භාවිත කරන ලදී. Final model selection සඳහා nested-CV performance, repeated-CV performance, RMSE, MAE, variability සහ overfitting evidence එකට සලකා බලන ලදී.

Final selected model exact fitted pipeline එකක් ලෙස save කරන ලදී. Subsequent SHAP, recommendation, validation සහ deployment steps වල model එක නැවත train නොකර මෙම exact saved artifact එක load කර භාවිත කරයි.

In [44]:
print(FINAL_MODEL_PATH.exists())
print(FINAL_MODEL_PATH)

True
d:\internship-employability-project\models\phase3\phase3_psycap_selected_model.pkl


In [45]:
print(
    final_model.get_params()
)

{'memory': None, 'steps': [('scaler', StandardScaler()), ('model', ElasticNet(alpha=np.float64(0.14873521072935117), l1_ratio=0.0, max_iter=50000,
           random_state=42))], 'transform_input': None, 'verbose': False, 'scaler': StandardScaler(), 'model': ElasticNet(alpha=np.float64(0.14873521072935117), l1_ratio=0.0, max_iter=50000,
           random_state=42), 'scaler__copy': True, 'scaler__with_mean': True, 'scaler__with_std': True, 'model__alpha': np.float64(0.14873521072935117), 'model__copy_X': True, 'model__fit_intercept': True, 'model__l1_ratio': 0.0, 'model__max_iter': 50000, 'model__positive': False, 'model__precompute': False, 'model__random_state': 42, 'model__selection': 'cyclic', 'model__tol': 0.0001, 'model__warm_start': False}


In [46]:
final_model_parameters = {
    key: value
    for key, value in final_model.get_params().items()
    if key.startswith("model__")
}

final_model_parameters

{'model__alpha': np.float64(0.14873521072935117),
 'model__copy_X': True,
 'model__fit_intercept': True,
 'model__l1_ratio': 0.0,
 'model__max_iter': 50000,
 'model__positive': False,
 'model__precompute': False,
 'model__random_state': 42,
 'model__selection': 'cyclic',
 'model__tol': 0.0001,
 'model__warm_start': False}

In [47]:
final_selection_decision = pd.DataFrame([
    {
        "Final_Model": FINAL_MODEL_NAME,
        "Nested_Mean_R2": final_winner_row["Nested_Mean_R2"],
        "Nested_Std_R2": final_winner_row["Nested_Std_R2"],
        "Repeated_CV_Mean_R2": final_winner_row["CV_Mean_R2"],
        "Repeated_CV_Std_R2": final_winner_row["CV_Std_R2"],
        "CV_Mean_MAE": final_winner_row["CV_Mean_MAE"],
        "CV_Mean_RMSE": final_winner_row["CV_Mean_RMSE"],
        "Train_Validation_Gap":
            final_winner_row["CV_Train_Validation_Gap"],
        "Overfitting_Risk":
            final_winner_row["Overfitting_Risk"],
        "Decision":
            "SELECTED"
    }
])

final_selection_decision

,Final_Model,Nested_Mean_R2,Nested_Std_R2,Repeated_CV_Mean_R2,Repeated_CV_Std_R2,CV_Mean_MAE,CV_Mean_RMSE,Train_Validation_Gap,Overfitting_Risk,Decision
0,ElasticNet,0.573876,0.077173,0.574259,0.088289,0.136181,0.229558,0.021995,Low,SELECTED


In [48]:
final_selection_decision.to_csv(
    REPORT_DIR / "Final_Selected_Model_Decision.csv",
    index=False
)

## Step 08 Final Model Selection

Hyperparameter tuning substantially reduced overfitting in the nonlinear models. In particular, SVR, Random Forest and XGBoost showed large reductions in train-validation gaps and improved cross-validated R² after tuning.

However, nested cross-validation showed that Elastic Net provided the strongest overall generalization performance among the evaluated models.

Elastic Net achieved the highest nested mean R² while maintaining low variability and a small train-validation gap. The optimized Elastic Net solution used an L1 ratio of zero, indicating that the data favored L2-style regularization rather than sparse L1 feature selection.

Therefore, Elastic Net was selected as the final Phase 03 predictive model.

The exact fitted preprocessing-and-model pipeline was saved and verified by reloading it and reproducing identical predictions. Subsequent explainability, recommendation and validation steps will use this saved artifact without retraining.